# mqdm Jupyter smoke test

This notebook is intentionally small. It exercises the current Rich-backed `mqdm` behavior in a notebook context before introducing any notebook-specific backend.

In [1]:
import asyncio
import logging
import time

import multiprocessing
multiprocessing.set_start_method("fork")

import mqdm

## Simple loop

In [2]:
for _ in mqdm.mqdm(range(30), desc="simple"):
    time.sleep(0.2)

Output()

## Nested loop with mqdm.print

In [8]:
for outer in mqdm.mqdm(range(3), desc="outer"):
    for inner in mqdm.mqdm(range(50), desc=lambda _, i: f"inner {outer}:{i}"):
        time.sleep(0.05)
    mqdm.print(f"outer={outer}")

Output()

outer=1

outer=2

## Logging

In [4]:
mqdm.install_logging(level=logging.INFO)
log = logging.getLogger("mqdm.notebook")

for i in mqdm.mqdm(range(3), desc="logging"):
    log.info("step %s", i)
    time.sleep(0.5)

mqdm.uninstall_logging()

Output()

00:52:48 | INFO | mqdm.notebook: step 0

00:52:48 | INFO | mqdm.notebook: step 1

00:52:49 | INFO | mqdm.notebook: step 2

## Thread pool

In [5]:
def example_fn(i):
    time.sleep(0.5)
    return i * 10

mqdm.pool(example_fn, range(10), desc="thread", pool_mode="thread", n_workers=2)

Output()

[0, 10, 20, 30, 40, 50, 60, 70, 80, 90]

## Process pool

In [16]:
def example_fn(i):
    time.sleep(0.5)
    mqdm.print(mqdm.utils.process_name())
    return i * 10

mqdm.pool(example_fn, range(10), desc="process", pool_mode="process", n_workers=2)

Output()

ForkProcess-9

ForkProcess-10

ForkProcess-9

ForkProcess-10

...

ForkProcess-10

ForkProcess-9

ForkProcess-9

ForkProcess-10

[0, 10, 20, 30, 40, 50, 60, 70, 80, 90]

In [15]:
import time
import mqdm
import random
from mqdm import print  # Important for cross-process printing

def example_fn(i, steps=26, delay=0.01):
    for _ in mqdm.mqdm(range(random.randint(1, steps)), desc=f"worker {i}", transient=True):
        time.sleep(delay)
        if random.random() < 0.01:
            print(f"Worker {i} is doing something important!")
    return i


import mqdm
from mqdm.executor import Initializer

from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed

def run_with_process_pool(xs, n_workers=4):
    print(xs)
    with (
        mqdm.mqdm(total=len(xs), desc="Very important work") as pbar,
        ProcessPoolExecutor(
            max_workers=n_workers,
            initializer=Initializer(pool_mode="process"),
        ) as executor,
    ):
        print("hiii")
        # display(pbar)
        futures = {executor.submit(example_fn, x): x for x in xs}
        for future in as_completed(futures):
            y = future.result()
            print(y)
            pbar.update(1)

print("hi")
run_with_process_pool([1,2,3,4,5,6])

hi

[1, 2, 3, 4, 5, 6]

Output()

hiii

1

...

3

2

4

5

6

In [14]:
type(mqdm._runtime.pbar)

mqdm.progress.ProgressProxy

In [13]:
type(mqdm._runtime.get_pbar("process"))

mqdm.progress.ProgressProxy

## Async iterator

In [11]:
async def source():
    for i in range(30):
        await asyncio.sleep(0.02)
        yield i

async def run_async_iter():
    async for i in mqdm.mqdm(source(), total=30, desc="async-iter"):
        await asyncio.sleep(0.02)
    mqdm.print(f"async item={i}")

await run_async_iter()

Output()

async item=29

## Async pool

In [12]:
async def work(i):
    await asyncio.sleep(0.5)
    return i * 10

results = await mqdm.apool(work, range(10), desc="apool", n_workers=2)
results

Output()

[0, 10, 20, 30, 40, 50, 60, 70, 80, 90]